# 03 — Prove Constraints Are NOT Enforced

If you're coming from Oracle, SQL Server, or Postgres, this notebook is going to feel wrong.

In those systems, a `PRIMARY KEY` constraint means the engine **rejects** duplicate inserts.
A `UNIQUE` constraint means `ORA-00001` or `ERROR: duplicate key value violates unique constraint`.
A `FOREIGN KEY` means you can't insert a child row without a matching parent.

In Databricks, **none of that happens.** The constraints exist in metadata, but the engine
won't stop a single bad row from landing. Let's prove it.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

## Baseline: confirm clean data

In [2]:
customer_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.customers").collect()[0].cnt
print(f"Customers before: {customer_count}")

# Grab an existing customer for duplicate tests
existing = spark.sql(f"SELECT customer_id, email FROM {UC_CATALOG}.{UC_SCHEMA}.customers LIMIT 1").collect()[0]
print(f"Will duplicate: customer_id={existing.customer_id}, email={existing.email}")

Customers before: 200
Will duplicate: customer_id=1, email=johnsonjoshua@example.org


## Test 1: Insert a duplicate `customer_id` (PK violation)

In Oracle/SQL Server, this would throw an immediate error. Here?

In [3]:
spark.sql(f"""
    INSERT INTO {UC_CATALOG}.{UC_SCHEMA}.customers
    VALUES ({existing.customer_id}, 'pk_violation@test.com', 'Duplicate', 'PKTest',
            '555-0000', 'Testville', 'TX', '2026-01-01')
""")

print("INSERT with duplicate customer_id succeeded — no error!")

INSERT with duplicate customer_id succeeded — no error!


## Test 2: Insert a duplicate `email` (UNIQUE violation)

Same story — no `ORA-00001`, no `duplicate key violates unique constraint`. Just silence.

In [4]:
spark.sql(f"""
    INSERT INTO {UC_CATALOG}.{UC_SCHEMA}.customers
    VALUES (99999, '{existing.email}', 'Duplicate', 'EmailTest',
            '555-0001', 'Testville', 'TX', '2026-01-01')
""")

print("INSERT with duplicate email succeeded — no error!")

INSERT with duplicate email succeeded — no error!


## Test 3: Insert an order with non-existent `customer_id` (FK violation)

In the RDBMS world, this is where you'd get `ORA-02291: integrity constraint violated - parent key not found`.
On Databricks, the orphan row goes right in.

In [5]:
spark.sql(f"""
    INSERT INTO {UC_CATALOG}.{UC_SCHEMA}.orders
    VALUES (99999, -1, 'Ghost Product', 1, 19.99, '2026-01-01')
""")

print("INSERT with non-existent customer_id succeeded — FK not enforced!")

INSERT with non-existent customer_id succeeded — FK not enforced!


## Test 4: Bulk insert the entire `new_customers_batch` (20 violations)

In [6]:
spark.sql(f"""
    INSERT INTO {UC_CATALOG}.{UC_SCHEMA}.customers
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch
""")

new_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.customers").collect()[0].cnt
print(f"Customers after bulk insert: {new_count} (was {customer_count})")
print("All 30 rows inserted — including 20 with constraint violations!")

Customers after bulk insert: 232 (was 200)
All 30 rows inserted — including 20 with constraint violations!


## Show the damage: duplicate `customer_id` values

In [7]:
spark.sql(f"""
    SELECT customer_id, COUNT(*) AS cnt
    FROM {UC_CATALOG}.{UC_SCHEMA}.customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
""").show(truncate=False)

+-----------+---+
|customer_id|cnt|
+-----------+---+
|8          |2  |
|68         |2  |
|148        |2  |
|11         |2  |
|165        |2  |
|145        |2  |
|12         |2  |
|1          |2  |
|167        |2  |
|198        |2  |
|21         |2  |
+-----------+---+



## Show the damage: duplicate `email` values

In [8]:
spark.sql(f"""
    SELECT email, COUNT(*) AS cnt
    FROM {UC_CATALOG}.{UC_SCHEMA}.customers
    GROUP BY email
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
""").show(truncate=False)

+----------------------------+---+
|email                       |cnt|
+----------------------------+---+
|clam@example.org            |2  |
|epruitt@example.net         |2  |
|johnsonjoshua@example.org   |2  |
|victorgonzalez@example.net  |2  |
|kevinmorrison@example.net   |2  |
|williamsmichelle@example.org|2  |
|kimberlydecker@example.org  |2  |
|bradmiller@example.net      |2  |
|lbanks@example.net          |2  |
|mhoward@example.org         |2  |
|vjones@example.net          |2  |
+----------------------------+---+



## Show the damage: orphaned orders (FK violation)

In [9]:
spark.sql(f"""
    SELECT o.order_id, o.customer_id, o.product_name
    FROM {UC_CATALOG}.{UC_SCHEMA}.orders o
    LEFT JOIN {UC_CATALOG}.{UC_SCHEMA}.customers c
      ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""").show(truncate=False)

+--------+-----------+-------------+
|order_id|customer_id|product_name |
+--------+-----------+-------------+
|99999   |-1         |Ghost Product|
+--------+-----------+-------------+



## Why does Databricks do this?

In Oracle or SQL Server, every INSERT goes through a **single engine** that checks
a B-tree index before committing. That works when you have one writer hitting one server.

Databricks is a distributed system — hundreds of executors writing Parquet files in parallel.
There's no single lock manager to ask "does this PK already exist?" Enforcing uniqueness
at write time would require a **full table scan on every INSERT**, which kills the throughput
advantage that's the whole point of the lakehouse architecture.

So the constraints serve a different purpose here:
- **Optimizer hints** — the query planner can skip unnecessary joins when it knows a key is unique
- **Documentation** — downstream tools and users can see the intended data model
- **BI tool compatibility** — tools like Tableau and Power BI read these to auto-build relationships

But enforcement? That's the **pipeline's job**, not the storage engine's.

### The mental model shift

| RDBMS (Oracle, SQL Server, Postgres) | Lakehouse (Databricks) |
|---------------------------------------|------------------------|
| Engine enforces constraints at write time | Engine trusts the writer — constraints are hints |
| Stored procedures gate all writes | Pipelines gate all writes |
| Triggers fire on bad data | DLT expectations fire on bad data |
| `ORA-00001` rejects the INSERT | `expect_or_fail` halts the pipeline |
| One writer, one server, one lock manager | Distributed writers, no central lock |

In the next notebook, we'll build enforcement manually with PySpark and SQL — patterns
that will feel familiar if you've written stored procedures or triggers before. Then in
notebook 05, we'll see the real answer: **DLT expectations** — the lakehouse equivalent
of engine-level constraint enforcement.